## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, sys, json
import numpy as np
import pandas as pd
import joblib
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

# ============================================================
# PATHS — change these once if your Drive layout differs
# ============================================================
RAW  = "/content/drive/MyDrive/Machine_learning_project/Data/Raw"
PROC = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
SRC  = "/content/drive/MyDrive/Machine_learning_project/src"
os.makedirs(PROC, exist_ok=True)

# --- shared spine: the SAME target & split the whole team uses ---
sys.path.append(SRC)
import common

target = common.load_target()                # id, price, log_price
train_ids, test_ids = common.load_split()    # the fixed 4926 / 1232 split

print("target:", target.shape, "| train:", len(train_ids), "| test:", len(test_ids))

target: (6158, 3) | train: 4926 | test: 1232


## Load data

In [3]:
listings = pd.read_csv(f"{RAW}/listings.csv")
reviews  = pd.read_csv(f"{RAW}/reviews.csv")
print("listings:", listings.shape, "| reviews:", reviews.shape)

listings: (6996, 79) | reviews: (575824, 6)


## Preprocess - one text row per spliting

In [4]:
'''
`reviews.csv` has **many rows per listing** (575k reviews / 6,109 listings).
It must be aggregated to **one row per listing** before joining — otherwise the
same listing's reviews could straddle the train/test boundary.

Note the key names: `listings.csv` uses `id`, `reviews.csv` uses `listing_id`.
We rename to `id` so everything in the project speaks one language.
'''
# (a) aggregate reviews -> one row per listing
rev = reviews.dropna(subset=["comments"]).copy()
rev["comments"] = rev["comments"].astype(str)

agg = (
    rev.groupby("listing_id")
       .agg(reviews_concat=("comments", lambda s: " ".join(s)),
            review_count=("comments", "size"))
       .reset_index()
       .rename(columns={"listing_id": "id"})          # match the spine's key
)

# (b) start from the USABLE listings (target defines who's in), attach text
#     how="left" keeps listings that have NO reviews (they'd vanish on an inner join)
text_df = (
    target[["id"]]
      .merge(listings[["id", "name", "description"]], on="id", how="left")
      .merge(agg, on="id", how="left")
)

# (c) fill gaps — 722 listings have no reviews (they are NEW listings, not junk)
text_df["reviews_concat"] = text_df["reviews_concat"].fillna("")
text_df["review_count"]   = text_df["review_count"].fillna(0).astype(int)
text_df["has_reviews"]    = (text_df["review_count"] > 0).astype(int)   # <- the signal
text_df["name"]           = text_df["name"].fillna("")
text_df["description"]    = text_df["description"].fillna("")

# (d) one combined text field per listing
text_df["full_text"] = (
    text_df["name"] + " " + text_df["description"] + " " + text_df["reviews_concat"]
).str.strip()

text_df = text_df.set_index("id")

# the canonical row order for EVERY matrix produced in this notebook
all_ids    = text_df.index.values
train_mask = np.isin(all_ids, train_ids)

print("text table:", text_df.shape)
print(text_df["has_reviews"].value_counts().to_dict(), "  (0 = no reviews)")

text table: (6158, 6)
{1: 5436, 0: 722}   (0 = no reviews)


## Features set A - TF-IDF

In [5]:
# Bag-of-words. Captures the literal vocabulary ("luxury", "penthouse", "shared").
# Fitted on training text only.
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,             # ignore words in <5 listings (noise)
    max_df=0.8,           # ignore words in >80% of listings (uninformative)
    stop_words="english",
    ngram_range=(1, 2),   # unigrams + bigrams
)

tfidf.fit(text_df.loc[train_ids, "full_text"])       # FIT: train only
X_tfidf_raw = tfidf.transform(text_df["full_text"])  # APPLY: all rows

# two hand-built features; scale the count (fit on train), keep the flag as 0/1
scaler = StandardScaler()
rc = text_df["review_count"].values.reshape(-1, 1).astype(float)
scaler.fit(rc[train_mask])                           # FIT: train only
rc_scaled = scaler.transform(rc)
hr = text_df["has_reviews"].values.reshape(-1, 1).astype(float)

X_tfidf = sparse.hstack([
    X_tfidf_raw,
    sparse.csr_matrix(np.hstack([rc_scaled, hr])),
]).tocsr()

print("TF-IDF matrix:", X_tfidf.shape)   # (6158, 5002)

TF-IDF matrix: (6158, 5002)


## Feature set B — sentence embeddings

In [6]:
'''
Captures **meaning** rather than exact words ("cozy" ≈ "snug").

The truncation problem: transformers read ~512 tokens, but some listings have
1,577 reviews. So we embed **two things separately**:

1. `name + description`  → 384 dims
2. each review individually, then **average** them → 384 dims

Averaging (rather than truncating one huge blob) lets every review contribute.
We cap at the **30 most recent** reviews per listing for speed and recency.

> **Turn the GPU on** first: Runtime → Change runtime type → GPU.
'''
!pip install sentence-transformers -q

In [7]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set")

token set


In [9]:

from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("embedding dim:", embedder.get_sentence_embedding_dimension())

device: cuda


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dim: 384


/tmp/ipykernel_2034/837692829.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dim:", embedder.get_sentence_embedding_dimension())


In [10]:
# --- (a) description embeddings: one per listing ---
desc_texts = (text_df["name"] + ". " + text_df["description"]).loc[all_ids].tolist()

desc_emb = embedder.encode(
    desc_texts, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
print("description embeddings:", desc_emb.shape)     # (6158, 384)

Batches:   0%|          | 0/97 [00:00<?, ?it/s]

description embeddings: (6158, 384)


In [11]:
# --- (b) review embeddings: embed each review, then average per listing ---
N_REVIEWS = 30

rv = reviews.dropna(subset=["comments"]).copy()
rv["comments"] = rv["comments"].astype(str)
rv["date"] = pd.to_datetime(rv["date"], errors="coerce")
rv = rv.sort_values("date", ascending=False)
rv = rv.groupby("listing_id").head(N_REVIEWS)        # most recent N per listing
rv = rv[rv["listing_id"].isin(all_ids)]
print("reviews to embed:", len(rv))

rev_vecs = embedder.encode(
    rv["comments"].tolist(), batch_size=128, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)

# average the vectors per listing; no-review listings get a ZERO vector
mean_rev   = pd.DataFrame(rev_vecs, index=rv["listing_id"].values).groupby(level=0).mean()
review_emb = mean_rev.reindex(all_ids).fillna(0.0).values
print("review embeddings:", review_emb.shape)        # (6158, 384)

reviews to embed: 123346


Batches:   0%|          | 0/964 [00:00<?, ?it/s]

review embeddings: (6158, 384)


In [12]:
# --- (c) assemble: [ desc 384 | reviews 384 | review_count | has_reviews ] ---
X_emb = np.hstack([desc_emb, review_emb, rc_scaled, hr])
print("embedding matrix:", X_emb.shape)              # (6158, 770)

embedding matrix: (6158, 770)


## Save everything

In [13]:
sparse.save_npz(f"{PROC}/text_tfidf.npz", X_tfidf)
np.save(f"{PROC}/text_emb.npy", X_emb)
np.save(f"{PROC}/text_ids.npy", all_ids)

# the fitted transforms — so they never need refitting, and can be interpreted
joblib.dump(tfidf,  f"{PROC}/text_tfidf_vectorizer.joblib")
joblib.dump(scaler, f"{PROC}/text_scaler.joblib")

# the two embedding halves, for a possible "description vs reviews" ablation
np.save(f"{PROC}/text_emb_desc.npy", desc_emb)
np.save(f"{PROC}/text_emb_reviews.npy", review_emb)

print("saved:")
print("  text_tfidf.npz ", X_tfidf.shape)
print("  text_emb.npy   ", X_emb.shape)
print("  text_ids.npy   ", all_ids.shape)

saved:
  text_tfidf.npz  (6158, 5002)
  text_emb.npy    (6158, 770)
  text_ids.npy    (6158,)


## Trial (not the real modelling)

In [14]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

row_of = {i: k for k, i in enumerate(all_ids)}
rows   = lambda ids: [row_of[i] for i in ids]

y = target.set_index("id")["log_price"]
y_train, y_test = y.loc[train_ids].values, y.loc[test_ids].values

m = Ridge(alpha=1.0).fit(X_tfidf[rows(train_ids)], y_train)
p = m.predict(X_tfidf[rows(test_ids)])
print(f"sanity — Ridge on TF-IDF   R2: {r2_score(y_test, p):.3f} | "
      f"log-RMSE: {np.sqrt(mean_squared_error(y_test, p)):.3f}")

sanity — Ridge on TF-IDF   R2: 0.586 | log-RMSE: 0.371
